# 02 · Climbing the Ladder

Five models, same split, same folds, same metric. The point is not to find the
best one — it is to see how little separates them, and to build the habit of
proving that a step up the ladder was worth taking.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT / "data" / "turbine_readings.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "turbine_clean.csv"
if not CLEAN.exists():
    raise SystemExit("Run Block 1 notebook 02 first — it produces turbine_clean.csv")
print("reading", CLEAN.name)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUM = ["age_years","rotor_speed_rpm","vibration_mm_s","gearbox_temp_c",
       "hours_per_year","vibration_missing"]
CAT = ["site"]
TARGET = "failed_within_90d"

df = pd.read_csv(CLEAN, parse_dates=["install_date"])
df["hours_per_year"] = df["operating_hours"] / df["age_years"].clip(lower=0.1)

X, y = df[NUM + CAT], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUM),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"train {X_train.shape}  test {X_test.shape}  failure rate {y.mean():.3f}")

## Rung 0 and 1 — the baselines

`DummyClassifier` is the majority-class baseline, written down so it appears in
the comparison rather than living in someone's memory.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

MODELS = {
    "0 majority class":     DummyClassifier(strategy="most_frequent"),
    "1 single tree (d=3)":  DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0),
    "2 logistic regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "3 random forest":      RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                   random_state=0, n_jobs=-1),
    "4 gradient boosting":  HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06,
                                                           max_iter=300, early_stopping=True,
                                                           class_weight="balanced", random_state=0),
}
print(f"{len(MODELS)} models on the ladder")

## Same folds, same metric, spread reported

Recall is the metric, chosen before looking at anything, because a missed failure
is the expensive error on this problem.

**One caution before reading the table.** Recall is measured at the default 0.5
threshold, and different model families produce differently-shaped probability
distributions. A model that is well calibrated at 0.5 will look better than one
that ranks just as well but concentrates its probabilities differently — which is
a fact about the threshold, not about the model.

So we report **average precision** alongside it. That one is threshold-free, and
it is the fair comparator between families.

In [ ]:
import time
rows = []
for name, clf in MODELS.items():
    pipe = Pipeline([("prep", make_prep()), ("clf", clf)])
    t0 = time.perf_counter()
    rec = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall", n_jobs=-1)
    ap  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    rows.append({"model": name,
                 "recall@0.5": rec.mean().round(3),
                 "recall_sd": rec.std().round(3),
                 "avg_precision": ap.mean().round(3),
                 "ap_sd": ap.std().round(3),
                 "fit_seconds": round(time.perf_counter() - t0, 1)})
ladder = pd.DataFrame(rows)
ladder

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].errorbar(ladder["recall@0.5"], ladder["model"], xerr=ladder["recall_sd"],
                 fmt="o", ms=9, capsize=5, lw=2)
axes[0].set_xlabel("recall at threshold 0.5 (+/- 1 sd)")
axes[0].set_title("Threshold-dependent — compare with care")
axes[1].errorbar(ladder["avg_precision"], ladder["model"], xerr=ladder["ap_sd"],
                 fmt="o", ms=9, capsize=5, lw=2, color="darkorange")
axes[1].set_xlabel("average precision (+/- 1 sd)")
axes[1].set_title("Threshold-free — the fair comparison")
plt.tight_layout()

### Read the error bars before the means

Where the intervals overlap heavily, the ranking is not established. Reporting the
top model as "best" in that situation is a claim the data does not support.

The honest sentence is usually: *these three are indistinguishable on this data,
so choose on cost, interpretability and operational burden instead.*

In [ ]:
# is the gap between the top two bigger than their variation?
top2 = ladder.nlargest(2, "avg_precision").reset_index(drop=True)
gap = top2.loc[0, "avg_precision"] - top2.loc[1, "avg_precision"]
noise = (top2.loc[0, "ap_sd"] + top2.loc[1, "ap_sd"]) / 2
print(f"{top2.loc[0,'model']}  vs  {top2.loc[1,'model']}")
print(f"gap between means : {gap:.3f}")
print(f"typical fold sd   : {noise:.3f}")
print(f"\n{'The gap is inside the noise — not a demonstrated difference.' if gap < noise else 'The gap exceeds typical fold variation — worth investigating further.'}")

## What each rung costs beyond its score

A score is one column of the decision. Here are the others.

In [ ]:
cost_table = pd.DataFrame([
 {"model":"logistic regression","explains itself":"yes — coefficients","tuning":"almost none",
  "serving":"trivial","drift response":"refit, seconds"},
 {"model":"random forest","explains itself":"partly — importances","tuning":"almost none",
  "serving":"easy","drift response":"refit, seconds"},
 {"model":"gradient boosting","explains itself":"partly — importances","tuning":"real",
  "serving":"easy","drift response":"refit and retune"},
])
cost_table

## Feature importance, and the trap in it

The default tree importance is biased toward high-cardinality features — a
continuous variable offers hundreds of split points, a binary flag offers one.
Permutation importance asks the question people think they are asking: does the
model actually rely on this?

In [ ]:
from sklearn.inspection import permutation_importance

rf = Pipeline([("prep", make_prep()),
               ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                              random_state=0, n_jobs=-1))]).fit(X_train, y_train)

r = permutation_importance(rf, X_test, y_test, n_repeats=10,
                           random_state=0, scoring="recall", n_jobs=-1)
imp = (pd.DataFrame({"feature": NUM + CAT,
                     "importance": r.importances_mean.round(4),
                     "sd": r.importances_std.round(4)})
       .sort_values("importance", ascending=False))
imp

---

## What this notebook established

- The majority-class baseline, written down
- Five models on identical folds with the same metric
- Whether the top gap survives fold variation — usually it does not
- The costs that a score does not capture
- Which features the model actually relies on, measured by permutation

## Your turn

1. Add `hours_per_year` to the feature list and re-run. Does any model move
   beyond its error bar?
2. Change the metric to `average_precision` and re-run. Does the ranking change?
   What does it mean if it does?
3. Drop the strongest feature from `imp` and re-run the ladder. How much of the
   performance was that one column?